In [19]:
import kagglehub
import pandas
# Download latest version
path = kagglehub.dataset_download("ealaxi/paysim1")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'paysim1' dataset.
Path to dataset files: /kaggle/input/paysim1


In [20]:
df = pd.read_csv(f"/kaggle/input/paysim1/PS_20174392719_1491204439457_log.csv")

In [21]:
df.shape  # way too big for me to do on colab
df = df.sample(300000, random_state=42) # can get similar results using a subsample
df.shape

(300000, 11)

In [22]:
df['isFraud'].value_counts()

,count
isFraud,
0,299612
1,388


In [23]:
df.isnull().sum()

,0
step,0
type,0
amount,0
nameOrig,0
oldbalanceOrg,0
newbalanceOrig,0
nameDest,0
oldbalanceDest,0
newbalanceDest,0
isFraud,0


In [24]:
def clean_dataset(df):
  df = df.drop(columns=["step", "nameOrig", "nameDest", "isFlaggedFraud"])
  df = pd.get_dummies(df, columns=["type"])
  df = df.reset_index(drop=True)
  return df
df = clean_dataset(df)
df.head()

,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,type_CASH_IN,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER
0,330218.42,20866.00,351084.42,452419.57,122201.15,0,True,False,False,False,False
1,11647.08,30370.00,18722.92,0.00,0.00,0,False,False,False,True,False
2,152264.21,106589.00,258853.21,201303.01,49038.80,0,True,False,False,False,False
3,1551760.63,0.00,0.00,3198359.45,4750120.08,0,False,False,False,False,True
4,78172.30,2921331.58,2999503.88,415821.90,337649.60,0,True,False,False,False,False


Dataset is now clean and ready for feature engineering


In [25]:
#Feature Engineering
def engineer_features(df):
  #original account
  df["balanceChangeOrg"] = df["oldbalanceOrg"] - df["newbalanceOrig"]
  df["isEmptied"] = df["newbalanceOrig"] == 0
  df["balanceDiffOrg"] = df["amount"] - df["balanceChangeOrg"]
  #destination account
  df["balanceChangeDest"] = df["newbalanceDest"] - df["oldbalanceDest"]
  df["balanceDiffDest"] = df["amount"] - df["balanceChangeDest"]
  return df
df = engineer_features(df)
df.head()

,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,type_CASH_IN,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER,balanceChangeOrg,isEmptied,balanceDiffOrg,balanceChangeDest,balanceDiffDest
0,330218.42,20866.00,351084.42,452419.57,122201.15,0,True,False,False,False,False,-330218.42,False,6.604368e+05,-330218.42,660436.84
1,11647.08,30370.00,18722.92,0.00,0.00,0,False,False,False,True,False,11647.08,False,-1.818989e-12,0.00,11647.08
2,152264.21,106589.00,258853.21,201303.01,49038.80,0,True,False,False,False,False,-152264.21,False,3.045284e+05,-152264.21,304528.42
3,1551760.63,0.00,0.00,3198359.45,4750120.08,0,False,False,False,False,True,0.00,True,1.551761e+06,1551760.63,0.00
4,78172.30,2921331.58,2999503.88,415821.90,337649.60,0,True,False,False,False,False,-78172.30,False,1.563446e+05,-78172.30,156344.60


Feature engineering complete, moving onto the pipeline now

In [26]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

X = df.drop("isFraud", axis = 1)
y = df["isFraud"]
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.2, random_state = 42)


In [27]:
##Building pipelines
rf_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestClassifier(random_state=42))
])

LR_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(random_state=42))
])

XGB_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', XGBClassifier(random_state=42))
])


In [53]:
#run pipelines
def run_pipelines(pipe, name, X_train, X_test, y_train, y_test):
  pipe.fit(X_train, y_train)
  predict= pipe.predict(X_test)
  report = classification_report(y_test, predict)
  print(f"\n {name}: \n{report}")
  return predict, report


In [55]:
#random forest
rf_predict, rf_report = run_pipelines(rf_pipe, "Random Forest", X_train, X_test, y_train, y_test)


 Random Forest: 
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     59930
           1       1.00      1.00      1.00        70

    accuracy                           1.00     60000
   macro avg       1.00      1.00      1.00     60000
weighted avg       1.00      1.00      1.00     60000



In [57]:
X_basic = df.drop(columns=["isFraud", "balanceChangeOrg", "isEmptied", "balanceDiffOrg", "balanceChangeDest", "balanceDiffDest"])
y = df["isFraud"]

X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(X_basic, y, test_size=0.2, random_state=42)

rf_predict_b, rf_report_b = run_pipelines(rf_pipe, "RF Basic", X_train_b, X_test_b, y_train_b, y_test_b)

# Testing model with and without engineered features
# to verify feature engineering impact on performance
# Without: Fraud recall 0.70 | With: Fraud recall 1.00
# Engineered features significantly improve fraud detection


 RF Basic: 
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     59930
           1       0.94      0.70      0.80        70

    accuracy                           1.00     60000
   macro avg       0.97      0.85      0.90     60000
weighted avg       1.00      1.00      1.00     60000



In [59]:
#Logistic Regression
lr_predict, lr_report = run_pipelines(LR_pipe, "Logistic Regression", X_train, X_test, y_train, y_test)


 Logistic Regression: 
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     59930
           1       0.86      0.46      0.60        70

    accuracy                           1.00     60000
   macro avg       0.93      0.73      0.80     60000
weighted avg       1.00      1.00      1.00     60000



In [60]:
#XGBoost
XGB_predict, XGB_report = run_pipelines(XGB_pipe, "XGBoost", X_train, X_test, y_train, y_test)


 XGBoost: 
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     59930
           1       0.68      0.71      0.69        70

    accuracy                           1.00     60000
   macro avg       0.84      0.86      0.85     60000
weighted avg       1.00      1.00      1.00     60000



**BEST: Random Forest (Perfect scores)**: Builds hundreds of decision trees, each looking at different parts of the data. The engineered features like isEmptied and balanceChangeOrg create clear decision boundaries that trees can split on perfectly eg. "if account emptied AND transfer type AND large amount → fraud." Multiple trees voting together catches every pattern.


---


**XGBoost (Decent but not perfect):** Builds trees sequentially, each one fixing the previous one's mistakes. With only 70 fraud cases in the test set and 300k rows of training data, XGBoost might be overfitting to the majority class. Its sequential approach works best with more balanced data or careful tuning, which is exactly what GridSearchCV will help with.

---


**WORST: Logistic Regression (Weakest):** Tries to draw a straight line between fraud and non-fraud. But fraud patterns here are complex, it's not just "high amount = fraud." It's combinations like high amount AND emptied account AND transfer type. LR can't capture those interactions without you manually creating them as features. Tree models handle this naturally.

---


**Key takeaway:** No single model wins every time. Complex, non-linear patterns favour tree-based models. Simple, linear relationships favour LR. Always test and compare.


Run gridsearchCV to find the best hyperparameters for each model

In [40]:
#gridsearchCV
from sklearn.model_selection import GridSearchCV

rf_params = {
    'model__n_estimators': [50, 100, 200],
    'model__max_depth': [5, 10, None]
}

xgb_params = {
    'model__n_estimators': [50, 100, 200],
    'model__max_depth': [3, 5, 10],
    'model__learning_rate': [0.01, 0.1, 0.3]
}

lr_params = {
    'model__C': [0.01, 0.1, 1, 10],
    'model__max_iter': [1000, 5000]
}

def tune_model(pipe, params, name, X_train, y_train):
  grid_search = GridSearchCV(pipe, params, cv=3, scoring='f1')
  grid_search.fit(X_train, y_train)
  print(f"The best parameters for the {name} model are: \n {grid_search.best_params_}")
  print(f"Thes best score for {name} is: \n {grid_search.best_score_}")
  return grid_search


In [34]:
tune_model(rf_pipe, rf_params, "Random Forest",X_train, y_train)

The best parameters for the Random Forest model are: 
 {'model__max_depth': 10, 'model__n_estimators': 50}
Thes best score for Random Forest is: 
 0.9984202211690363


In [35]:
tune_model(LR_pipe, lr_params, "Logistic Regression",X_train, y_train)

The best parameters for the Logistic Regression model are: 
 {'model__C': 10, 'model__max_iter': 1000}
Thes best score for Logistic Regression is: 
 0.6760317460317461


In [46]:
xgb_grid = tune_model(XGB_pipe, xgb_params, "XGBoost", X_train, y_train)

The best parameters for the XGBoost model are: 
 {'model__learning_rate': 0.1, 'model__max_depth': 3, 'model__n_estimators': 100}
Thes best score for XGBoost is: 
 0.9984202211690363
